# Module 6 — The Context Layer

**Course:** Build Your First AI Agent from Scratch  
**Community:** [Autonomous MSP](https://skool.com/autonomous-msp-2162)

**What you build in this module:**
- An MCP Standards Server (FastAPI) — the single source of truth for your network standards
- An MCP client the agent queries before every run
- A Network Graph Store (Neo4j or demo) — your topology as a queryable graph
- A `get_topology_context()` function that tells the agent what's connected to what

**Why it matters:** Without context, your agent knows networking in general but not YOUR network. It doesn't know your OSPF process ID, your change freeze window, or that rtr-01 failing affects 14 services.

## Step 1 — Install Dependencies

In [ ]:
!pip install fastapi uvicorn requests -q
# neo4j driver is optional — we use DEMO_MODE=True in this notebook
# !pip install neo4j -q

## Part 1 — The MCP Standards Server

MCP (Model Context Protocol) is just a local FastAPI server that the agent queries to get current standards and operational state. You maintain this file. It is your single source of truth.

The agent queries it at the start of every run: *"What are our OSPF process IDs? Is there a change freeze active right now?"*

In [ ]:
# ── MCP Server code ─────────────────────────────────────────────
# In production: run this as a separate service
#   python mcp_server.py
# Then set DEMO_MODE=False in the client below.

MCP_SERVER_CODE = '''
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional
import uvicorn
import datetime

app = FastAPI(title="Network Agent MCP Server", version="1.0")

# ── UPDATE THESE TO MATCH YOUR NETWORK ────────────────────────
STANDARDS = {
    "routing": {
        "primary_protocol": "OSPF",
        "ospf_process_id":  1,
        "backbone_area":    "0.0.0.0",
        "site_area_mapping": {
            "datacenter-1": "0.0.0.0",
            "datacenter-2": "0.0.0.1",
            "branch-sites": "0.0.0.2",
        },
        "prohibited_protocols": ["EIGRP", "RIPv2"],
    },
    "interfaces": {
        "default_mtu":           1500,
        "core_link_mtu":         9000,
        "ospf_hello_interval":   10,
        "ospf_dead_interval":    40,
    },
}


def is_change_freeze_active() -> bool:
    now = datetime.datetime.now()
    # Annual holiday freeze: Dec 15 → Jan 10
    return (now.month == 12 and now.day >= 15) or (now.month == 1 and now.day <= 10)


@app.get("/mcp/standards")
def get_standards():
    return STANDARDS


@app.get("/mcp/state/change-window")
def get_change_window():
    frozen = is_change_freeze_active()
    return {
        "is_frozen":    frozen,
        "frozen_until": "January 10" if frozen else None,
        "reason":       "Annual holiday freeze" if frozen else None,
    }


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8080)
'''

# Save the server code to a file (so you can run it outside Colab)
with open("mcp_server.py", "w") as f:
    f.write(MCP_SERVER_CODE)

print("mcp_server.py saved.")
print("To run the server: !python mcp_server.py")
print("Then set DEMO_MODE=False in the MCP client below.")

## Step 2 — The MCP Client

The agent uses this client to query the MCP server. `DEMO_MODE=True` returns hardcoded standards — no server needed for testing. Set `DEMO_MODE=False` when your server is running.

The key output is `get_mcp_context_for_prompt()` — a formatted text block the agent injects into every prompt.

In [ ]:
import requests

MCP_BASE_URL = "http://localhost:8080"
DEMO_MODE    = True  # Set to False when mcp_server.py is running

# Fallback data used when DEMO_MODE=True or server is unreachable
DEMO_STANDARDS = {
    "routing": {
        "primary_protocol": "OSPF",
        "ospf_process_id":  1,
        "backbone_area":    "0.0.0.0",
    },
    "change_freeze": False,
}


def get_standards() -> dict:
    if DEMO_MODE:
        return DEMO_STANDARDS
    try:
        return requests.get(f"{MCP_BASE_URL}/mcp/standards", timeout=5).json()
    except Exception:
        return DEMO_STANDARDS  # fallback if server is down


def is_change_freeze_active() -> bool:
    if DEMO_MODE:
        return DEMO_STANDARDS["change_freeze"]
    try:
        return requests.get(f"{MCP_BASE_URL}/mcp/state/change-window", timeout=5).json()["is_frozen"]
    except Exception:
        return False


def get_mcp_context_for_prompt() -> str:
    """
    Format MCP data as a text block for injection into the agent prompt.
    The agent reads this to know YOUR network standards — not generic ones.
    """
    standards = get_standards()
    frozen    = is_change_freeze_active()
    routing   = standards.get("routing", {})

    lines = [
        "NETWORK STANDARDS (from MCP server):",
        f"  Routing protocol:  {routing.get('primary_protocol', 'OSPF')}",
        f"  OSPF process ID:   {routing.get('ospf_process_id', 1)}",
        f"  Backbone area:     {routing.get('backbone_area', '0.0.0.0')}",
    ]

    if frozen:
        lines.append("")
        lines.append("  !! CHANGE FREEZE IS ACTIVE — do not propose or apply any changes !!")

    return "\n".join(lines)


print("=== MCP Context Block (what the agent sees) ===")
print(get_mcp_context_for_prompt())

## Part 2 — The Network Knowledge Graph

A knowledge graph models your network as nodes (devices, services) and relationships (connections). This lets the agent answer questions like:
- "What is connected to rtr-01?"
- "If rtr-01 fails, what services are affected?"

**DEMO_MODE=True** uses a hardcoded Python dict — no Neo4j needed.  
**DEMO_MODE=False** queries a real Neo4j database.

In [ ]:
# Demo topology — represents what you'd store in Neo4j
DEMO_TOPOLOGY = {
    "devices": {
        "rtr-01": {"role": "core-router",          "site": "dc1", "ip": "192.168.1.1"},
        "rtr-02": {"role": "distribution-router",   "site": "dc1", "ip": "192.168.1.2"},
        "rtr-03": {"role": "branch-router",          "site": "branch-1", "ip": "10.0.0.1"},
        "fw-01":  {"role": "firewall",               "site": "dc1", "ip": "192.168.1.254"},
    },
    "connections": [
        {"from": "rtr-01", "to": "rtr-02", "interface_a": "Gi0/0", "interface_b": "Gi0/0", "type": "core-link"},
        {"from": "rtr-01", "to": "rtr-03", "interface_a": "Gi0/1", "interface_b": "Gi0/0", "type": "wan-link"},
        {"from": "fw-01",  "to": "rtr-01", "interface_a": "Gi0/0", "interface_b": "Gi0/2", "type": "security-link"},
    ],
    "services": {
        "web-portal": {"host": "rtr-02", "ip": "192.168.10.100", "critical": True},
        "erp-system":  {"host": "rtr-02", "ip": "192.168.10.200", "critical": True},
        "monitoring":  {"host": "rtr-01", "ip": "192.168.1.50",   "critical": False},
    },
}

print(f"Demo topology loaded: {len(DEMO_TOPOLOGY['devices'])} devices, "
      f"{len(DEMO_TOPOLOGY['connections'])} connections, "
      f"{len(DEMO_TOPOLOGY['services'])} services")

## Step 3 — NetworkGraphStore

Two key methods:
- `get_device_neighbors()` — what is this device directly connected to?
- `get_blast_radius()` — if this device fails, what is affected?

Both run against DEMO_TOPOLOGY now. When you install Neo4j, flip `DEMO_MODE = False` and uncomment the real queries.

In [ ]:
import os

GRAPH_DEMO_MODE = True  # Set to False when Neo4j is running


class NetworkGraphStore:
    """Query network topology from Neo4j (or demo data)."""

    def __init__(self):
        if not GRAPH_DEMO_MODE:
            from neo4j import GraphDatabase
            uri      = os.environ.get("NEO4J_URI",  "bolt://localhost:7687")
            user     = os.environ.get("NEO4J_USER", "neo4j")
            password = os.environ.get("NEO4J_PASSWORD", "password")
            self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def get_device_neighbors(self, device_name: str) -> list:
        """What devices is this device directly connected to?"""
        if GRAPH_DEMO_MODE:
            neighbors = []
            for conn in DEMO_TOPOLOGY["connections"]:
                if conn["from"] == device_name:
                    neighbors.append({
                        "neighbor":         conn["to"],
                        "local_interface":  conn["interface_a"],
                        "remote_interface": conn["interface_b"],
                        "link_type":        conn["type"],
                    })
                elif conn["to"] == device_name:
                    neighbors.append({
                        "neighbor":         conn["from"],
                        "local_interface":  conn["interface_b"],
                        "remote_interface": conn["interface_a"],
                        "link_type":        conn["type"],
                    })
            return neighbors

        # Real Neo4j query (when GRAPH_DEMO_MODE = False)
        with self.driver.session() as session:
            result = session.run("""
                MATCH (d:Device {name: $name})-[r:CONNECTED_TO]-(n:Device)
                RETURN n.name as neighbor, r.local_interface, r.remote_interface
            """, name=device_name)
            return [dict(record) for record in result]

    def get_blast_radius(self, device_name: str) -> dict:
        """If this device fails, what devices and services are affected?"""
        if GRAPH_DEMO_MODE:
            affected_devices  = []
            affected_services = []

            # Find all directly connected devices
            for conn in DEMO_TOPOLOGY["connections"]:
                if conn["from"] == device_name:
                    affected_devices.append(conn["to"])
                elif conn["to"] == device_name:
                    affected_devices.append(conn["from"])

            # Find services hosted on affected devices (or the failed device itself)
            for svc_name, svc in DEMO_TOPOLOGY["services"].items():
                if svc["host"] in affected_devices or svc["host"] == device_name:
                    affected_services.append({
                        "service":  svc_name,
                        "ip":       svc["ip"],
                        "critical": svc["critical"],
                    })

            return {
                "failed_device":              device_name,
                "directly_affected_devices":   affected_devices,
                "affected_services":           affected_services,
                "critical_services_affected":  [s for s in affected_services if s["critical"]],
            }

    def get_topology_context(self, device_name: str) -> str:
        """Format topology data as a text block for the agent prompt."""
        neighbors = self.get_device_neighbors(device_name)
        blast     = self.get_blast_radius(device_name)

        lines = [f"TOPOLOGY CONTEXT for {device_name}:"]

        if neighbors:
            lines.append("  Directly connected to:")
            for n in neighbors:
                lines.append(
                    f"    - {n['neighbor']} via "
                    f"{n['local_interface']} <-> {n['remote_interface']} ({n['link_type']})"
                )
        else:
            lines.append("  No topology data found for this device.")

        if blast and blast["critical_services_affected"]:
            lines.append("  CRITICAL services that depend on this device:")
            for svc in blast["critical_services_affected"]:
                lines.append(f"    - {svc['service']} ({svc['ip']})")

        return "\n".join(lines)


# Test it
graph = NetworkGraphStore()

print("=== Neighbors of rtr-01 ===")
for n in graph.get_device_neighbors("rtr-01"):
    print(f"  {n}")

print("\n=== Blast radius if rtr-01 fails ===")
blast = graph.get_blast_radius("rtr-01")
print(f"  Affected devices:  {blast['directly_affected_devices']}")
print(f"  Critical services: {[s['service'] for s in blast['critical_services_affected']]}")

## Step 4 — Wire Both Context Sources Into a Single Prompt Block

In your agent's `observe_node`, call both MCP and the graph store, then combine the results into one context string that gets injected into the prompt.

In [ ]:
def build_full_context(device_name: str = None) -> str:
    """
    Combines MCP standards + topology context into one block.
    This is what gets prepended to every agent prompt.
    """
    mcp_context  = get_mcp_context_for_prompt()
    topo_context = ""

    if device_name:
        topo_context = graph.get_topology_context(device_name)

    # Join non-empty blocks with a blank line separator
    parts = [p for p in [mcp_context, topo_context] if p]
    return "\n\n".join(parts)


print("=== Full context block for rtr-01 ===")
print(build_full_context(device_name="rtr-01"))

## Step 5 — Neo4j Setup (When You're Ready)

The easiest way to run Neo4j locally is Docker. Run this once, then set `GRAPH_DEMO_MODE = False` above.

In [ ]:
# ── Neo4j with Docker ──────────────────────────────────────────
# Run this in your terminal (NOT in Colab — Colab can't run Docker):

DOCKER_COMMAND = """
docker run -d \
  --name neo4j \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  neo4j:5
"""

SEED_CYPHER = """
// Create device nodes
MERGE (:Device {name: 'rtr-01', role: 'core-router',   site: 'dc1'})
MERGE (:Device {name: 'rtr-02', role: 'dist-router',   site: 'dc1'})
MERGE (:Device {name: 'fw-01',  role: 'firewall',      site: 'dc1'})

// Create connections
MATCH (a:Device {name: 'rtr-01'}), (b:Device {name: 'rtr-02'})
MERGE (a)-[:CONNECTED_TO {local_interface: 'Gi0/0', remote_interface: 'Gi0/0'}]->(b)

// Create services
MERGE (:Service {name: 'web-portal', ip: '192.168.10.100', critical: true})
MATCH (s:Service {name: 'web-portal'}), (d:Device {name: 'rtr-02'})
MERGE (s)-[:HOSTED_ON]->(d)
"""

print("Docker command to start Neo4j:")
print(DOCKER_COMMAND)
print("\nBrowse to http://localhost:7474 (user: neo4j, password: password)")
print("\nSeed Cypher (run in the Neo4j browser):")
print(SEED_CYPHER)

## What's Next

Your agent now knows YOUR network. In **Module 7** you add production safety: enhanced approval gates, LangSmith tracing, Prometheus metrics, and the deployment checklist.

---

**Module Challenge:** Add a `get_blast_radius` tool to your tool registry using the `NetworkGraphStore.get_blast_radius()` method. Test it by asking: "What happens to services if rtr-01 fails?"